# RalphGuard — Candidate Training & Validation (Run All)

Notebook นี้เป็นจุดสั่งงานเดียวสำหรับเตรียมข้อมูล, เพิ่มคลังสมุนไพรไทย, ตรวจ data leakage, ฝึก Candidate ของ 4 endpoint เดิม, ฝึก Skin Dryness research candidate และสร้างรายงาน validation/กราฟ

> กติกาหลัก: ไม่สร้าง label จากการไม่พบอันตราย, ไม่ใช้ prediction เก่าเป็น label, ไม่แทนสารสกัดสมุนไพรด้วย SMILES เดียว และไม่เขียนทับโมเดล production โดยอัตโนมัติ

## ขั้นตอนที่ 1 — ตั้งค่าและตรวจ environment

ค่าเริ่มต้นถูกตั้งให้กด **Run All** ได้ทันที การฝึก 4 endpoint ใช้ข้อมูลสูงสุด 15,000 molecular identities ต่อ endpoint โดยเก็บหลักฐานทดลอง/ตรวจทานไว้ก่อน weak labels เสมอ

In [7]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys
import pandas as pd
from IPython.display import Image, Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'data_prep.py').exists() and (candidate / 'docker-compose.yml').exists():
            return candidate
    raise FileNotFoundError('กรุณาเปิด Notebook จากภายใน RalphGuard repository')

ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scientific'))

RUN_ALL = True
SEED_THAI_HERBS = True
IMPORT_NICEATM_HPPT = True
TRAIN_EXISTING_ENDPOINTS = True
TRAIN_SKIN_DRYNESS = True
REBUILD_ECHA_FROM_RAW_PDF = False  # True ต้องใช้อินเทอร์เน็ต; False ใช้ cache ที่ตรวจ hash แล้ว
REFRESH_SKIN_DRYNESS_LITERATURE = False  # cached, source-hashed PubMed mapping is Run-All default
REFRESH_ICSC_DISCOVERY = False  # cached audited discovery is Run-All default
ICSC_MAX_CARDS = 100  # increase only for a deliberate full-network refresh
MAX_TRAINING_ROWS_PER_ENDPOINT = 15_000
EXISTING_ENDPOINT_VALIDATION_PROFILE = 'auto'
DRYNESS_VALIDATION_PROFILE = 'full'

CANDIDATE_V2_DIR = ROOT / 'scientific' / 'models' / 'candidate_v2'
CANDIDATE_V3_DIR = ROOT / 'scientific' / 'models' / 'candidate_v3'
print('Project root:', ROOT)
print('Python:', sys.version.split()[0])
print('Run mode: full candidate workflow')

Project root: C:\Users\suraw\Documents\GitHub\ralphguard
Python: 3.11.15
Run mode: full candidate workflow


## ขั้นตอนที่ 2 — แหล่งข้อมูลและคลังสมุนไพรไทย

- Skin Sensitization: NICEATM Human Predictive Patch Test (HPPT)
- Skin Dryness weak positive: EUH066 จาก CLP Annex VI พร้อม CAS/InChIKey/PubChem
- Skin Dryness direct labels: งานทดลอง TEWL/skin hydration ที่ระบุผลบวกหรือลบชัดเจน
- สมุนไพรไทย: botanical/material records จาก Thai Herbal Pharmacopoeia; whole extract ไม่เข้า single-molecule QSAR

In [ ]:
from scripts.import_echa_euh066 import build_evidence
from scripts.import_icsc_skin_dryness import import_icsc
from scripts.import_skin_dryness_literature_expansion import import_literature

stage_status = {}
echa_frame, echa_report = build_evidence(refresh=REBUILD_ECHA_FROM_RAW_PDF)
stage_status['echa_euh066'] = {
    'resolved_structures': int(len(echa_frame)),
    'source_sha256': echa_report['source_sha256'],
    'label_role': 'regulatory weak positive',
}

literature_report = import_literature(refresh=REFRESH_SKIN_DRYNESS_LITERATURE)
stage_status['skin_dryness_literature'] = {
    'development_training_eligible': literature_report['development']['training_eligible'],
    'new_explicit_negatives': literature_report['development']['explicit_negatives'],
    'review_required': literature_report['development']['review_required'],
    'external_unique': literature_report['external']['unique_identities'],
    'external_positive': literature_report['external']['positive'],
    'external_negative': literature_report['external']['negative'],
    'exact_overlap': len(literature_report['external']['development_exact_identity_overlap']),
}
icsc_rows, icsc_report = import_icsc(
    refresh=REFRESH_ICSC_DISCOVERY, max_cards=ICSC_MAX_CARDS
)
stage_status['icsc_discovery'] = {
    'cards_screened': icsc_report['cards_screened'],
    'review_candidates': icsc_report['standardized_phrase_matches'],
    'resolved_structures': icsc_report['resolved_structures'],
    'training_eligible': icsc_report['training_eligible'],
}

if IMPORT_NICEATM_HPPT:
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'import_niceatm_hppt.py')], cwd=ROOT, check=True)
hppt_report_path = ROOT / 'data' / 'curated' / 'sens_hppt_import_report.json'
hppt_report = json.loads(hppt_report_path.read_text(encoding='utf-8'))
stage_status['niceatm_hppt'] = {
    key: hppt_report[key]
    for key in ('raw_test_rows', 'accepted_unique_structures', 'label_1', 'label_0')
}

if SEED_THAI_HERBS:
    # Read the catalogue size back from the seed script's JSON output instead
    # of repeating the number here, which silently went stale when the
    # catalogue grew from 30 to 107 plants.
    seed_herbs = subprocess.run(
        [sys.executable, str(ROOT / 'backend' / 'scripts' / 'seed_thai_herbs.py')],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    if seed_herbs.returncode == 0:
        stage_status['thai_herbs'] = {
            'status': 'seeded',
            **json.loads(seed_herbs.stdout.strip().splitlines()[-1]),
        }
    else:
        stage_status['thai_herbs'] = {
            'status': 'database_unavailable',
            'note': 'เปิด PostgreSQL/Docker แล้วรันเซลล์นี้ซ้ำ; ขั้น model training ทำต่อได้',
            'stderr_tail': seed_herbs.stderr.strip().splitlines()[-1:] if seed_herbs.stderr else [],
        }

display(pd.DataFrame(stage_status).T)

## ขั้นตอนที่ 3 — Identity, provenance และ leakage audit

RDKit สร้าง canonical SMILES/InChIKey, รวมชื่อพ้องที่เป็นโมเลกุลเดียวกัน, ตัด same-tier conflict และกัน exact identity ของ external holdout ออกจากชุดฝึก Label 0 ต้องเป็น explicit negative เท่านั้น

In [ ]:
from scripts.skin_dryness_workflow import prepare_evidence_pool

candidate_paths = [
    ROOT / 'data' / 'staging' / 'skin_dryness_candidates.csv',
    ROOT / 'data' / 'staging' / 'skin_dryness_literature_review.csv',
    ROOT / 'data' / 'staging' / 'skin_dryness_literature_expansion.csv',
    ROOT / 'data' / 'staging' / 'skin_dryness_icsc_candidates.csv',
    ROOT / 'data' / 'curated' / 'skin_dryness_echa_euh066.csv',
]
missing = [str(path.relative_to(ROOT)) for path in candidate_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing Skin Dryness evidence files: {missing}')

dryness_candidates = pd.concat([pd.read_csv(path) for path in candidate_paths], ignore_index=True, sort=False)
external_path = ROOT / 'data' / 'external' / 'skin_dryness.csv'
dryness_external = pd.read_csv(external_path) if external_path.exists() else pd.DataFrame()
dryness_audit = prepare_evidence_pool(
    dryness_candidates,
    external_holdout=dryness_external,
    write_outputs=True,
)
dryness_manifest = dryness_audit['manifest']
display(pd.Series(dryness_manifest['counts'], name='count').to_frame())
display(
    dryness_audit['training']
    .groupby(['label_quality', 'normalized_label'], dropna=False)
    .size()
    .rename('rows')
    .to_frame()
)
assert dryness_manifest['minimum_pool_met'], 'Evidence pool ต้องมีอย่างน้อย 10,000 unique structures'
assert dryness_manifest['supervised_fit_ready'], 'ยังไม่มี verified positive และ explicit negative ครบทั้งสอง class'

## ขั้นตอนที่ 4 — ปรับ Candidate ของ 4 endpoint เดิม

ใช้ endpoint-specific Morgan/MACCS/descriptors, evidence-quality weights, class-balanced optimization และ soft voting ของ Random Forest + Extra Trees + Logistic Regression + HistGradientBoosting Threshold เลือกจาก OOF predictions และรายงาน scaffold CV แยกต่างหาก

In [ ]:
if TRAIN_EXISTING_ENDPOINTS:
    command = [
        sys.executable, str(ROOT / 'scripts' / 'train_candidate_v2.py'),
        '--validation-profile', EXISTING_ENDPOINT_VALIDATION_PROFILE,
        '--max-training-rows-per-endpoint', str(MAX_TRAINING_ROWS_PER_ENDPOINT),
    ]
    subprocess.run(command, cwd=ROOT, check=True)

candidate_v2_report = json.loads((CANDIDATE_V2_DIR / 'validation_report.json').read_text(encoding='utf-8'))
legacy_rows = []
for endpoint, item in candidate_v2_report['endpoints'].items():
    legacy_rows.append({
        'endpoint': endpoint,
        'n': item['dataset']['n'],
        'positive': item['dataset']['positive'],
        'negative': item['dataset']['negative'],
        'feature_mode': item['dataset']['feature_mode'],
        'OOF_AUC': item['candidate_oof'].get('auc'),
        'OOF_MCC': item['candidate_oof'].get('mcc'),
        'Scaffold_AUC': item['candidate_scaffold_grouped'].get('auc'),
        'Scaffold_MCC': item['candidate_scaffold_grouped'].get('mcc'),
        'External_status': item['external'].get('status'),
    })
legacy_metrics = pd.DataFrame(legacy_rows).set_index('endpoint')
display(legacy_metrics)

## ขั้นตอนที่ 5 — Benchmark และฝึก Skin Dryness research candidate

เปรียบเทียบ `morgan`, `maccs_descr`, `morgan_maccs_descr` โดยให้ scaffold MCC สำคัญกว่า random OOF AUC แล้วฝึก Candidate-v3 เท่านั้น หากจำนวน true negatives หรือ external holdout ยังต่ำกว่าเกณฑ์ โมเดลจะเปิดเป็น research preview พร้อมคำเตือนและถูกห้าม promote

In [8]:
from scripts.skin_dryness_workflow import train_candidate_from_notebook

dryness_report = None
if TRAIN_SKIN_DRYNESS:
    dryness_report = train_candidate_from_notebook(
        ROOT / 'data' / 'curated' / 'skin_dryness_training.csv',
        validation_profile=DRYNESS_VALIDATION_PROFILE,
        output_dir=CANDIDATE_V3_DIR,
    )
else:
    report_path = CANDIDATE_V3_DIR / 'skin_dryness_validation_report.json'
    dryness_report = json.loads(report_path.read_text(encoding='utf-8')) if report_path.exists() else None

if dryness_report:
    display(pd.Series({
        'n': dryness_report['n'],
        'positive': dryness_report['positive'],
        'negative': dryness_report['negative'],
        'selected_features': dryness_report['feature_benchmark']['selected_feature_mode'],
        'OOF_accuracy': dryness_report['candidate_oof'].get('accuracy'),
        'OOF_balanced_accuracy': dryness_report['candidate_oof'].get('balanced_accuracy'),
        'OOF_AUC': dryness_report['candidate_oof'].get('auc'),
        'OOF_MCC': dryness_report['candidate_oof'].get('mcc'),
        'Scaffold_accuracy': dryness_report['candidate_scaffold_grouped'].get('accuracy'),
        'Scaffold_balanced_accuracy': dryness_report['candidate_scaffold_grouped'].get('balanced_accuracy'),
        'Scaffold_AUC': dryness_report['candidate_scaffold_grouped'].get('auc'),
        'Scaffold_MCC': dryness_report['candidate_scaffold_grouped'].get('mcc'),
        'External_status': dryness_report['external'].get('status'),
        'External_accuracy': dryness_report['external'].get('metrics', {}).get('accuracy'),
        'External_balanced_accuracy': dryness_report['external'].get('metrics', {}).get('balanced_accuracy'),
        'All_validation_levels_balanced_accuracy_at_least_85%': dryness_report['accuracy_target']['all_validation_levels_meet_target'],
        'Promotion': dryness_report['promotion_status'],
    }))
    display(pd.Series(dryness_report['promotion_checks'], name='passed').to_frame())

n                                       34
positive                                26
negative                                 8
selected_features              maccs_descr
OOF_AUC                              0.692
OOF_MCC                              0.403
Scaffold_AUC                         0.726
Scaffold_MCC                         0.403
External_status                   complete
Promotion            research_only_blocked
dtype: object

,passed
training_positive_at_least_25,True
training_negative_at_least_25,False
scaffold_validation_complete,True
scaffold_mcc_at_least_0_40,True
scaffold_balanced_accuracy_at_least_0_65,True
external_validation_complete,True
external_positive_at_least_15,False
external_negative_at_least_15,False
external_auc_at_least_0_75,False
external_mcc_at_least_0_40,False


## ขั้นตอนที่ 6 — สรุปผล, กราฟ และ deployment gate

Candidate artifacts อยู่ใน `scientific/models/candidate_v2` และ `candidate_v3` Production เดิมไม่ถูกแก้ Worker จะ hot-load Skin Dryness เฉพาะ artifact ที่มีสถานะ `research_preview` และผล API/UI ต้องแสดงว่าเป็นโมเดลทดลอง

In [ ]:
summary = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'run_all_completed': True,
    'production_models_modified': False,
    'thai_herbs': stage_status.get('thai_herbs'),
    'niceatm_hppt': stage_status.get('niceatm_hppt'),
    'echa_euh066': stage_status.get('echa_euh066'),
    'skin_dryness_literature': stage_status.get('skin_dryness_literature'),
    'icsc_discovery': stage_status.get('icsc_discovery'),
    'skin_dryness_manifest': dryness_manifest,
    'candidate_v2_endpoints': legacy_rows,
    'skin_dryness_candidate': dryness_report,
    'next_runtime_behavior': (
        'Skin Dryness is hot-loaded as a clearly marked research candidate on the next assessment; '
        'automatic production promotion is forbidden.'
    ),
}
summary_path = ROOT / 'scientific' / 'models' / 'notebook_run_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

display(Markdown(f'''
### Run All เสร็จแล้ว

- รายงานรวม: `{summary_path.relative_to(ROOT)}`
- Candidate เดิม 4 endpoint: `{CANDIDATE_V2_DIR.relative_to(ROOT)}`
- Skin Dryness research candidate: `{CANDIDATE_V3_DIR.relative_to(ROOT)}`
- Production models modified: **No**
- Skin Dryness promotion: **{dryness_report['promotion_status'] if dryness_report else 'not trained'}**
'''))

# ── ก่อน/หลัง การทำความสะอาดข้อมูล ────────────────────────────────────────
# ทุกช่องคือจำนวนแถวที่ "เหลืออยู่" หลังผ่านขั้นนั้น ไม่ใช่จำนวนที่ถูกตัด
# จึงไล่ตรวจย้อนได้ทีละขั้นว่าข้อมูลหายไปตรงไหนและเพราะอะไร
FUNNEL_TH = {
    'raw_rows_loaded': '1 โหลดจากทุกแหล่ง',
    'valid_structure_and_label': '2 โครงสร้าง+label ใช้ได้',
    'external_holdout_quarantined': '3 กัน external holdout',
    'best_evidence_tier_only': '4 เก็บชั้นหลักฐานดีสุด',
    'same_tier_conflicts_removed': '5 ตัด label ขัดแย้ง',
    'deduplicated_to_identities': '6 รวมสารซ้ำ',
    'class_balanced_training_set': '7 ชุดฝึกหลังปรับสมดุลคลาส',
}
cleaning_rows = []
for endpoint, item in candidate_v2_report['endpoints'].items():
    stats = item.get('dataset', {})
    funnel = {stage['stage']: stage['rows'] for stage in stats.get('cleaning_funnel', [])}
    if not funnel:
        continue
    loaded = funnel.get('raw_rows_loaded', 0)
    kept = funnel.get('class_balanced_training_set', 0)
    row = {'endpoint': endpoint}
    row.update({label: funnel.get(key) for key, label in FUNNEL_TH.items()})
    row['เหลือ %'] = round(100.0 * kept / max(1, loaded), 2)
    row['pos:neg ก่อน'] = stats.get('eligible_positive_negative_ratio_before_cap')
    positives = stats.get('retained_positive_identities') or 0
    negatives = stats.get('retained_negative_identities') or 0
    row['pos:neg หลัง'] = round(positives / negatives, 1) if negatives else None
    cleaning_rows.append(row)

if cleaning_rows:
    display(Markdown(
        '### ก่อน/หลัง การทำความสะอาดข้อมูล\n\n'
        'ตัวเลข = จำนวนแถวที่เหลือหลังผ่านแต่ละขั้น'
    ))
    display(pd.DataFrame(cleaning_rows).set_index('endpoint'))

# ── กราฟทั้งเส้นทาง: นำเข้า → ทำความสะอาด → ฝึก → ประเมิน ───────────────────
display(Markdown(
    '### กราฟทั้งเส้นทาง\n\n'
    '1. นำข้อมูลเข้าและ provenance · 2. ทำความสะอาด/ปรับสมดุลคลาส · '
    '3. โปรไฟล์ชุดฝึก · 4. ผล validation รายชั้น · 5. สรุปรวม'
))
plot_candidates = [
    CANDIDATE_V2_DIR / 'plots' / '00_algorithm_pipeline.png',
    CANDIDATE_V2_DIR / 'plots' / '00_data_ingestion.png',
]
for endpoint in candidate_v2_report['endpoints']:
    endpoint_plots = CANDIDATE_V2_DIR / 'plots' / endpoint
    plot_candidates += [
        endpoint_plots / '00_cleaning_funnel.png',
        endpoint_plots / '01_data_profile.png',
        endpoint_plots / '02_oof_validation.png',
        endpoint_plots / '04_scaffold_cv.png',
        endpoint_plots / '05_external_validation.png',
        endpoint_plots / '06_model_comparison.png',
    ]
plot_candidates.append(CANDIDATE_V2_DIR / 'plots' / '08_pipeline_summary.png')
plot_candidates += [
    CANDIDATE_V3_DIR / 'plots' / 'skin_dryness' / '01_feature_benchmark.png',
    CANDIDATE_V3_DIR / 'plots' / 'skin_dryness' / '02_oof_validation.png',
    CANDIDATE_V3_DIR / 'plots' / 'skin_dryness' / '03_scaffold_validation.png',
    CANDIDATE_V3_DIR / 'plots' / 'skin_dryness' / '04_external_validation.png',
]
for plot_path in plot_candidates:
    if plot_path.exists():
        display(Markdown(f'**{plot_path.relative_to(ROOT)}**'))
        display(Image(filename=str(plot_path), width=1100))

## แหล่งอ้างอิงหลัก

- ECHA, *Table of harmonised entries in Annex VI to CLP*: https://echa.europa.eu/information-on-chemicals/annex-vi-to-clp
- NICEATM, *Human Predictive Patch Test Database*: https://ntp.niehs.nih.gov/iccvam/methods/immunotox/hppt
- Mohammed et al., TEWL and skin penetration enhancers, PMID 24063883: https://pubmed.ncbi.nlm.nih.gov/24063883/
- Abrams et al., organic solvents and human skin barrier, PMID 8409532: https://pubmed.ncbi.nlm.nih.gov/8409532/
- Lodén and Wessman, 20% glycerin and TEWL, PMID 18498456: https://pubmed.ncbi.nlm.nih.gov/18498456/
- Pavlačková et al., panthenol formulation attribution, PMID 29577586: https://pubmed.ncbi.nlm.nih.gov/29577586/
- Pinto et al., 1,3-propanediol, hydration and TEWL, PMID 37699769: https://pubmed.ncbi.nlm.nih.gov/37699769/
- van der Valk et al., irritants and water vapour loss, PMID 4028973: https://pubmed.ncbi.nlm.nih.gov/4028973/
- PubMed source-held-out Skin Dryness evidence: PMID 37950377, 8565486, and 15941007
- ILO/WHO, International Chemical Safety Cards: https://chemicalsafety.ilo.org/dyn/icsc/showcard.home
- Thai Herbal Pharmacopoeia, Department of Medical Sciences: https://bdn-thp.dmsc.moph.go.th/

ข้อจำกัด: internal OOF/scaffold CV ไม่ใช่ clinical accuracy; Skin Dryness ชุดปัจจุบันมี explicit negatives และ independent external chemicals น้อย จึงเปิดได้เฉพาะ research preview จนกว่า promotion checks ทุกข้อจะผ่าน